In [116]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

from util import create_region_graph, convert_to_xarray, aggregate_regions, extract_population
from constants import target_regions, region_mapping

from imagematerials.constants import _IMAGE_REGIONS as image_regions

In [ ]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/loadfactor.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0) #

IMAGE_SE_original = pd.read_csv("image_3_4_data/SSP2_SE_IMAGE.csv", delimiter=";")
load_original = load_original.drop(["Unnamed: 2", "Unnamed: 3"], axis = 1)

,Unnamed: 0,Unnamed: 1,bus,train,car,highspeedtrain,airplane
1,1971,1,24.22732,32.51845,1.687623,310.333,124
2,1971,2,23.49061,31.52963,1.664334,310.333,124
3,1971,3,28.8396,38.70916,1.825295,310.333,124
4,1971,4,31.7182,42.57288,1.90514,310.333,124
5,1971,5,30.70826,41.21732,1.877599,310.333,124
...,...,...,...,...,...,...,...
3376,2100,22,19.27647,25.8733,1.522651,310.333,124
3377,2100,23,17.87558,23.99301,1.471821,310.333,124
3378,2100,24,17.16131,23.03429,1.445059,310.333,124
3379,2100,25,19.94757,26.77408,1.546281,310.333,124


In [ ]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


<xarray.DataArray 'load' (region: 26, year: 2, mode: 5)> Size: 2kB
array([[[124.0, 26.17416, 1.747353, 310.333, 35.13155],
        [124.0, 21.48092, 1.598681, 310.333, 28.83217]],

       [[124.0, 20.59569, 1.568691, 310.333, 27.644],
        [124.0, 18.26099, 1.486018, 310.333, 24.51031]],

       [[124.0, 23.68307, 1.670456, 310.333, 31.78794],
        [124.0, 20.20304, 1.555161, 310.333, 27.11697]],

       [[124.0, 25.38689, 1.723504, 310.333, 34.07485],
        [124.0, 20.38047, 1.561293, 310.333, 27.35513]],

       [[124.0, 37.28201, 2.048862, 310.333, 50.04074],
        [124.0, 29.24685, 1.836849, 310.333, 39.25578]],

       [[124.0, 30.46306, 1.870837, 310.333, 40.8882],
        [124.0, 22.78359, 1.641602, 310.333, 30.58065]],

       [[124.0, 27.11573, 1.775365, 310.333, 36.39534],
        [124.0, 21.87215, 1.611718, 310.333, 29.35729]],
...
       [[124.0, 27.00988, 1.772243, 310.333, 36.25326],
        [124.0, 22.04131, 1.617316, 310.333, 29.58434]],

       [[124.0, 27.01397, 1.772363, 310.333, 36.25875],
        [124.0, 22.88019, 1.644731, 310.333, 30.7103]],

       [[124.0, 24.3226, 1.690606, 310.333, 32.64633],
        [124.0, 20.10151, 1.55164, 310.333, 26.98069]],

       [[124.0, 28.1029, 1.804164, 310.333, 37.72035],
        [124.0, 23.10757, 1.652066, 310.333, 31.0155]],

       [[124.0, 19.96091, 1.546746, 310.333, 26.79198],
        [124.0, 18.38763, 1.490646, 310.333, 24.6803]],

       [[124.0, 35.04767, 1.992666, 310.333, 47.04177],
        [124.0, 28.83508, 1.825167, 310.333, 38.70309]],

       [[124.0, 21.35115, 1.594328, 310.333, 28.658],
        [124.0, 19.29521, 1.523317, 310.333, 25.89846]]], dtype=object)
Coordinates:
  * region   (region) object 208B 'BRA' 'CAN' 'CEU' 'CHN' ... 'USA' 'WAF' 'WEU'
  * year     (year) int64 16B 2019 2060
  * mode     (mode) object 40B 'airplane' 'bus' 'car' 'highspeedtrain' 'train'

In [119]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")

In [ ]:
# Convert the region names in xr_image_output to match the target regions
knowledge_graph = create_region_graph()
xr_regions = pkmGlobal.coords["region"].values  # Extract current region names

xr_region_filter = knowledge_graph.find_relations_inverse(xr_regions, target_regions)

In [121]:
pkmGlobal_target_regions = aggregate_regions(pkmGlobal, xr_region_filter) #km
tkmGlobal_target_regions = aggregate_regions(tkmGlobal, xr_region_filter) # million km
load_target_regions = aggregate_regions(load, xr_region_filter, aggregation = 'mean') #person/vehicle

In [ ]:
pop_yearly_million = pd.read_csv("image_3_4_data/pop_yearly.csv", delimiter=";", skiprows=3)
pop_yearly_million = pop_yearly_million.rename(columns=lambda x: int(x.replace("class_", "")) if "class_" in x else x)
pop_yearly_million = pop_yearly_million.set_index("t").stack().reset_index()
pop_yearly_million.columns = ["year", "region", "value"]  # Rename columns
pop_yearly_million = (
        pop_yearly_million.astype({"year": int, "region": int})  # Convert to integers
        .query("region not in [27, 28] and year in [2019, 2020, 2060]")  # Filter regions and years
    )
region_mapping = { i+1:region for i, region in enumerate(image_regions)}
pop_yearly_million["region"] = pop_yearly_million["region"].map(region_mapping)
pop_yearly_million_xr = pop_yearly_million.set_index(["region", "year"]).to_xarray()["value"]
pop_image_regions = pop_yearly_million_xr * 1e6 # from million to single people

pop = aggregate_regions(pop_image_regions, xr_region_filter)

In [ ]:
tkmGlobal_target_regions = tkmGlobal_target_regions * 1e6 #from million tkm to tkm


In [124]:
pkm_per_capita = pkmGlobal_target_regions / pop
tkm_per_capita = tkmGlobal_target_regions / pop

In [125]:
# Assign names to the DataArrays
pkm_per_capita.name = 'pkm_per_capita'
tkm_per_capita.name = 'tkm_per_capita'
load_target_regions.name = 'load'

# Step 1: Convert xarrays to DataFrames
pkm_per_capita_df = pkm_per_capita.to_dataframe().reset_index()
tkm_per_capita_df = tkm_per_capita.to_dataframe().reset_index()
load_df = load_target_regions.to_dataframe().reset_index()

c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\variable.py:338: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [126]:
# Melt the pkm_per_capita DataFrame
pkm_per_capita_df_melted = pkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["pkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for pkm_per_capita
pkm_per_capita_df_melted["variable"] = "pkm_per_capita"

# Melt the tkm_per_capita DataFrame
tkm_per_capita_df_melted = tkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["tkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for tkm_per_capita
tkm_per_capita_df_melted["variable"] = "tkm_per_capita"

# Merge the two melted DataFrames
merged_df = pd.merge(pkm_per_capita_df_melted, tkm_per_capita_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

load_df_melted = load_df.melt(id_vars=["region", "year", "mode"], 
                              value_vars=["load"], 
                              var_name="variable", value_name="value")

# Assign the variable name for load
load_df_melted["variable"] = "load"

# Merge the two melted DataFrames
merged_df = pd.merge(merged_df, load_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

# Optionally, you may want to combine or rearrange columns as needed
#merged_df = merged_df[['region', 'year', 'mode', 'variable', 'value']]

merged_df.to_csv("output_data.csv", index=False)


In [132]:
merged_df[(merged_df['variable']=='load')&(merged_df['mode']=='car')]

,region,year,mode,variable,value
3,Canada,2019,car,load,1.568691
21,Canada,2060,car,load,1.486018
39,Central Europe,2019,car,load,1.670456
57,Central Europe,2060,car,load,1.555161
75,China,2019,car,load,1.723504
93,China,2060,car,load,1.561293
111,India,2019,car,load,1.870837
129,India,2060,car,load,1.641602
147,Japan,2019,car,load,1.59474
165,Japan,2060,car,load,1.52431


In [ ]:
IMAGE_SE = IMAGE_SE_original[["Region","Variable","2020","2060"]]
IMAGE_SE.columns = IMAGE_SE.columns.str.lower()  # Convert column names to lowercase
total_pkm_tkm = IMAGE_SE[(IMAGE_SE['variable']=='Energy Service|Transportation|Passenger') | # billion pkm/yr
                    (IMAGE_SE['variable']=='Energy Service|Transportation|Freight')] # billion tkm/yr



total_pkm_tkm_melted = total_pkm_tkm.melt(id_vars=["region", "variable"], var_name="year", value_name="value")

total_pkm_tkm_melted = total_pkm_tkm_melted[total_pkm_tkm_melted["region"] != "World"]
total_pkm_tkm_melted["year"] = total_pkm_tkm_melted["year"].astype(int)

# Convert from billion km to km
total_pkm_tkm_melted["value"] *= 1e9

pkm_tkm_xr = total_pkm_tkm_melted.set_index(["region","year","variable"]).to_xarray()#["value"]

pkm_tkm_xr = pkm_tkm_xr.assign_coords(region=[xr_region_filter[r] for r in pkm_tkm_xr["region"].values])

pkm_tkm_xr = pkm_tkm_xr.groupby("region").sum()


pkm_tkm_per_capita = pkm_tkm_xr / pop

pkm_tkm_per_capita_df = pkm_tkm_per_capita.to_dataframe().reset_index()

pkm_tkm_per_capita_df.to_csv("pkm_tkm_per_capita_df.csv", index=False)

pkm_tkm_per_capita_df


,year,variable,region,value
0,2020,Energy Service|Transportation|Freight,Canada,74613.255505
1,2020,Energy Service|Transportation|Freight,Central Europe,41339.204121
2,2020,Energy Service|Transportation|Freight,China,14986.067197
3,2020,Energy Service|Transportation|Freight,India,2328.151150
4,2020,Energy Service|Transportation|Freight,Japan,40113.441836
5,2020,Energy Service|Transportation|Freight,Latin America,11080.117245
6,2020,Energy Service|Transportation|Freight,Middle East and Northern Africa,12891.815295
7,2020,Energy Service|Transportation|Freight,Other Asia,11245.877945
8,2020,Energy Service|Transportation|Freight,Other OECD,43154.722042
9,2020,Energy Service|Transportation|Freight,Reforming Economies,11405.911548
